# PhoBERT DoRA Benchmark

Notebook này tổng hợp benchmark Full Fine-Tuning, LoRA và DoRA cho PhoBERT trên UIT-VSFC. Logic train chính nằm trong `train.py`; notebook chỉ gọi script hoặc đọc `results/benchmark_results.csv`.

In [ ]:
from pathlib import Path
import itertools
import os
import shlex
import subprocess
import sys

if sys.platform.startswith('win'):
    import asyncio
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RESULTS = ROOT / 'results' / 'benchmark_results.csv'
ROOT, RESULTS

## GPU diagnostics

Run this cell before training. It checks whether the active notebook kernel has a CUDA-enabled PyTorch build, not just whether the machine has an NVIDIA GPU.

In [ ]:
import platform
import torch

print('Python:', sys.executable)
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__)
print('PyTorch CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('CUDA device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    major, minor = torch.cuda.get_device_capability(0)
    print('Compute capability:', f'{major}.{minor}')
else:
    print('This environment is using CPU-only PyTorch or cannot access CUDA.')
    print('Install a CUDA-enabled PyTorch wheel in this exact environment:')
    print(f'{sys.executable} -m pip install --upgrade --index-url https://download.pytorch.org/whl/cu129 torch torchvision torchaudio')

## Download model and dataset

Set `DOWNLOAD_ASSETS = True` and run this cell once when you have internet. Hugging Face will cache PhoBERT locally, so later training runs can use the local cache. If your network blocks Hugging Face, download will fail here before training starts.

In [ ]:
DOWNLOAD_ASSETS = False
MODEL_NAME = 'vinai/phobert-base-v2'
DATASET_NAME = 'uitnlp/vietnamese_students_feedback'
LOCAL_FILES_ONLY = False
FORCE_DOWNLOAD = False

if DOWNLOAD_ASSETS:
    os.environ.setdefault('HF_HUB_DISABLE_SYMLINKS_WARNING', '1')
    from datasets import load_dataset
    from huggingface_hub import snapshot_download
    from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer

    try:
        model_cache_dir = snapshot_download(
            repo_id=MODEL_NAME,
            local_files_only=LOCAL_FILES_ONLY,
            force_download=FORCE_DOWNLOAD,
        )
        tokenizer = AutoTokenizer.from_pretrained(model_cache_dir, use_fast=False, local_files_only=True)
        config = AutoConfig.from_pretrained(model_cache_dir, num_labels=3, local_files_only=True)
        model = AutoModelForSequenceClassification.from_pretrained(
            model_cache_dir,
            config=config,
            local_files_only=True,
            ignore_mismatched_sizes=True,
        )
        dataset = load_dataset(DATASET_NAME, trust_remote_code=True)
        print('PhoBERT cache:', model_cache_dir)
        print('Tokenizer vocab size:', len(tokenizer))
        print('Model loaded:', type(model).__name__)
        print(dataset)
    except Exception as exc:
        print('Download failed.')
        print('Common causes: no internet, firewall blocks huggingface.co, or missing dependency from requirements.txt.')
        print('Original error:')
        raise exc
else:
    print('Set DOWNLOAD_ASSETS = True to download PhoBERT and UIT-VSFC into the local Hugging Face cache.')

## Run benchmark commands

Cell này luôn có thể chạy một lệnh kiểm tra trực tiếp (`python train.py --help`). Để train thật, set `RUN_TRAINING = True`. Nên bắt đầu với `TRAIN_MODE = 'single'` hoặc `'smoke'` trước khi chạy benchmark đầy đủ.

In [ ]:
RUN_DIRECT_CHECK = True
RUN_TRAINING = True
TRAIN_MODE = 'benchmark'  # 'single', 'smoke', or 'benchmark'

seeds = [42, 43, 44]
ranks = [8, 16]
smoke_args = ['--epochs', '1', '--max-train-samples', '64', '--max-eval-samples', '64']

def printable_command(cmd):
    return ' '.join(shlex.quote(str(part)) for part in cmd)

def run_command(cmd):
    cmd = list(cmd)
    if Path(str(cmd[0])).name.lower().startswith('python') or str(cmd[0]).lower().endswith('python.exe'):
        if len(cmd) == 1 or cmd[1] != '-u':
            cmd.insert(1, '-u')
    print('$', printable_command(cmd), flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        env=env,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)
    return return_code

def check_accelerate():
    check_cmd = [sys.executable, '-m', 'pip', 'show', 'accelerate']
    completed = subprocess.run(check_cmd, cwd=ROOT, text=True, capture_output=True)
    if completed.returncode == 0:
        print('accelerate is installed for this kernel Python.')
        return True
    install_cmd = [sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt')]
    print('Missing accelerate in this kernel Python.')
    print('Run this command in a notebook cell or terminal:')
    print(printable_command(install_cmd))
    return False

direct_check_command = [sys.executable, 'train.py', '--help']

single_train_command = [
    sys.executable, 'train.py', '--method', 'dora', '--rank', '8', '--alpha', '16',
    '--dropout', '0.05', '--seed', '42', '--output-dir', 'outputs',
    '--epochs', '1', '--max-train-samples', '64', '--max-eval-samples', '64',
]

benchmark_commands = []
for seed in seeds:
    benchmark_commands.append([sys.executable, 'train.py', '--method', 'ft', '--seed', str(seed), '--output-dir', 'outputs'])
    for method, rank in itertools.product(['lora', 'dora'], ranks):
        benchmark_commands.append([
            sys.executable, 'train.py', '--method', method, '--rank', str(rank), '--alpha', str(2 * rank),
            '--dropout', '0.05', '--seed', str(seed), '--output-dir', 'outputs'
        ])

if TRAIN_MODE == 'single':
    commands = [single_train_command]
elif TRAIN_MODE == 'smoke':
    commands = [cmd + smoke_args for cmd in benchmark_commands[:3]]
elif TRAIN_MODE == 'benchmark':
    commands = benchmark_commands
else:
    raise ValueError("TRAIN_MODE must be 'single', 'smoke', or 'benchmark'")

runtime_ready = check_accelerate()
if RUN_DIRECT_CHECK:
    run_command(direct_check_command)

print('\nTraining commands:')
for cmd in commands:
    print(printable_command(cmd))

if RUN_TRAINING:
    if not runtime_ready:
        raise RuntimeError('Install dependencies first, then rerun this cell.')
    for cmd in commands:
        run_command(cmd)
else:
    print('\nSet RUN_TRAINING = True to execute the training commands above.')

## Aggregate results

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.read_csv(RESULTS) if RESULTS.exists() else pd.DataFrame()
if df.empty or df.dropna(how='all').empty:
    print(f'No benchmark rows found yet: {RESULTS}')
    print('Run train.py or set RUN_TRAINING = True above, then rerun this notebook.')
else:
    df = df.dropna(subset=['method'], how='any')
display(df.head())

In [ ]:
if df.empty:
    summary = pd.DataFrame()
else:
    summary = (
        df.groupby(['method', 'rank'], dropna=False)
        .agg(
            accuracy_mean=('accuracy', 'mean'), accuracy_std=('accuracy', 'std'),
            macro_f1_mean=('macro_f1', 'mean'), macro_f1_std=('macro_f1', 'std'),
            weighted_f1_mean=('weighted_f1', 'mean'), weighted_f1_std=('weighted_f1', 'std'),
            trainable_percent_mean=('trainable_percent', 'mean'),
            peak_vram_mb_mean=('peak_vram_mb', 'mean'),
            train_time_sec_mean=('train_time_sec', 'mean'),
            checkpoint_size_mb_mean=('checkpoint_size_mb', 'mean'),
            runs=('run_id', 'count'),
        )
        .reset_index()
        .sort_values(['method', 'rank'])
    )
display(summary)

## Plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df.empty:
    print('No benchmark rows to plot yet.')
else:
    plot_df = df.copy()
    plot_df['rank_label'] = plot_df['rank'].fillna('full').astype(str)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    sns.barplot(data=plot_df, x='method', y='macro_f1', hue='rank_label', ax=axes[0])
    axes[0].set_title('Macro-F1')
    sns.barplot(data=plot_df, x='method', y='trainable_percent', hue='rank_label', ax=axes[1])
    axes[1].set_title('Trainable params (%)')
    sns.barplot(data=plot_df, x='method', y='peak_vram_mb', hue='rank_label', ax=axes[2])
    axes[2].set_title('Peak VRAM (MB)')
    plt.tight_layout()

## DoRA merge check

In [ ]:
import torch
from torch import nn
from src.peft import DoRALinear

torch.manual_seed(0)
linear = nn.Linear(8, 4)
dora = DoRALinear.from_linear(linear, rank=2, alpha=4, dropout=0.0)
dora.lora_B.data.normal_(0, 0.02)
x = torch.randn(3, 8)
merged = dora.merge()
max_diff = (dora(x) - merged(x)).abs().max().item()
max_diff